In [50]:
# AVALIAÇÃO: Modelagem Matemática e Computacional

###**Integrantes do grupo:**
- Thiago Soalheiro Diamantino // RM: 569316
- Kauan Damasceno de Lima // RM: 573727

In [51]:
# Tema: Global Solution CitizenTech: Mobilidade Urbana Inteligente

Neste notebook reconstruímos a distribuição atual da frota de ônibus de uma cidade a partir de indicadores consolidados (Etapa 1) e propomos uma política de redistribuição representada por uma transformação linear (Etapa 2). Cada passo traz a explicação antes do código.

In [52]:
import numpy as np
import pandas as pd
from fractions import Fraction as F

REGIOES = ["Norte", "Sul", "Leste", "Oeste"]

# ETAPA 1: Diagnóstico e modelagem da situação atual

### a) Variáveis necessárias do problema
Queremos saber quantos ônibus estão em cada centro operacional de acordo com o relatório I passado sobre os registros operacionais da frota:

- $N$: ônibus na região **Norte**
- $S$: ônibus na região **Sul**
- $L$: ônibus na região **Leste**
- $O$: ônibus na região **Oeste**

Como são contagens de veículos, $N, S, L, O$ são **inteiros não negativos**.

### b) Modelo matemático
Cada indicador do Relatório I vira uma equação linear:

| Indicador do relatório | Equação |
|---|---|
| A frota regular tem 72 ônibus (soma total) | $N + S + L + O = 72$ |
| Norte + Oeste + o dobro do Sul = 76 | $N + 2S + O = 76$ |
| Sul + o dobro do Leste = 52 | $S + 2L = 52$ |
| Norte + Leste + o dobro do Oeste = 76 | $N + L + 2O = 76$ |

Como as equações relacionam as variáveis, formam um **sistema linear 4 × 4**.

### c) Representação matricial
O sistema pode ser escrito como $A\,\mathbf{x} = \mathbf{b}$, com $\mathbf{x} = [N, S, L, O]^T$:

$$
\begin{bmatrix}
1 & 1 & 1 & 1\\
1 & 2 & 0 & 1\\
0 & 1 & 2 & 0\\
1 & 0 & 1 & 2
\end{bmatrix}
\begin{bmatrix} N\\ S\\ L\\ O \end{bmatrix}
=
\begin{bmatrix} 72\\ 76\\ 52\\ 76 \end{bmatrix}
$$

Cada linha de $A$ é uma equação e cada **coluna** é uma região (a primeira coluna são os coeficientes de $N$, e assim por diante).

In [53]:
A = np.array([[1, 1, 1, 1],
              [1, 2, 0, 1],
              [0, 1, 2, 0],
              [1, 0, 1, 2]], dtype=float)

b = np.array([72, 76, 52, 76], dtype=float)

tabela = pd.DataFrame(A.astype(int), columns=REGIOES,
                      index=["(1) total", "(2) N+2S+O", "(3) S+2L", "(4) N+L+2O"])
tabela["b (Termo Independente)"] = b.astype(int)
tabela

,Norte,Sul,Leste,Oeste,b (Termo Independente)
(1) total,1,1,1,1,72
(2) N+2S+O,1,2,0,1,76
(3) S+2L,0,1,2,0,52
(4) N+L+2O,1,0,1,2,76


## d) A distribuição é única?
Para verificar se a distribuição de veículos é única, utilizamos o teorema de Rouché-Capelli. Analisamos o posto da matriz dos coeficientes $A$ em comparação com o posto da matriz ampliada $[A\vert{}b]$.

Como ambos os postos são iguais a 4 (o mesmo número de incógnitas) e o determinante da matriz $A$ é diferente de zero ($\det(A) = 3$):

* O sistema é classificado como possível e determinado.   
* Existe uma única solução para o problema.

In [54]:
posto_A = np.linalg.matrix_rank(A) # serve para calcular a característica de uma matriz de forma automática
posto_Ab = np.linalg.matrix_rank(np.column_stack([A, b]))
det_A = np.linalg.det(A) # serve para calcular o determinante da matriz dos coeficientes

print(f"Posto(A)     = {posto_A}")
print(f"Posto([A|b]) = {posto_Ab}")
print(f"Incógnitas   = 4")
print(f"det(A)       = {det_A:.4f}")

if posto_A == posto_Ab == 4 and abs(det_A) > 1e-9:
    print("\nConclusão: Sistema possível e determinado -> existe UMA ÚNICA distribuição de veículos.")

Posto(A)     = 4
Posto([A|b]) = 4
Incógnitas   = 4
det(A)       = 3.0000

Conclusão: Sistema possível e determinado -> existe UMA ÚNICA distribuição de veículos.


**Interpretação**: Os dados disponíveis são suficientes e consistentes para determinar uma única distribuição de veículos, indicando que não há equações redundantes ou contraditórias.

## e) Resolução em Python
Resolvemos $A\mathbf{x} = \mathbf{b}$ com `np.linalg.solve`, que calcula $\mathbf{x}$ sem inverter a matriz explicitamente (é mais estável numericamente).

In [55]:
x = np.linalg.solve(A, b)
x_atual = np.rint(x).astype(int)

print("Solução bruta:", x)
pd.DataFrame({"Região": REGIOES, "Ônibus (atual)": x_atual})

Solução bruta: [12. 20. 16. 24.]


,Região,Ônibus (atual)
0,Norte,12
1,Sul,20
2,Leste,16
3,Oeste,24


**Distribuição encontrada:** Norte = 12, Sul = 20, Leste = 16 e Oeste = 24 ônibus.

## f) Verificação
Verificar se os resultados obtidos satisfazem todos os registros operacionais fornecidos.

**Resposta:** Substituímos a solução no sistema original e conferimos também as restrições do problema: valores **inteiros** e **não negativos**.

In [56]:
resultado = A @ x_atual
print("A · x =", resultado.astype(int))
print("b     =", b.astype(int))

print("\nConferência equação por equação:")
print("(1) N+S+L+O  =", x_atual.sum(),                                  "(esperado 72)")
print("(2) N+2S+O   =", x_atual[0] + 2*x_atual[1] + x_atual[3],         "(esperado 76)")
print("(3) S+2L     =", x_atual[1] + 2*x_atual[2],                      "(esperado 52)")
print("(4) N+L+2O   =", x_atual[0] + x_atual[2] + 2*x_atual[3],         "(esperado 76)")
print("\nOK: todos os registros são satisfeitos, com valores inteiros e não negativos.")

A · x = [72 76 52 76]
b     = [72 76 52 76]

Conferência equação por equação:
(1) N+S+L+O  = 72 (esperado 72)
(2) N+2S+O   = 76 (esperado 76)
(3) S+2L     = 52 (esperado 52)
(4) N+L+2O   = 76 (esperado 76)

OK: todos os registros são satisfeitos, com valores inteiros e não negativos.


# ETAPA 2: Política de redistribuição

## a) Quantos ônibus cada região precisa?
Cada ônibus atende 40 passageiros no período. Então a necessidade de uma região é

$$\text{ônibus necessários} = \left\lceil \frac{\text{passageiros estimados}}{40} \right\rceil$$

Obs: Arredondamos para cima porque não existe fração de ônibus e cada passageiro precisa de uma vaga.

In [57]:
passageiros = np.array([560, 760, 720, 840])
capacidade = 40

necessarios = np.ceil(passageiros / capacidade).astype(int)
saldo = x_atual - necessarios

tabela2 = pd.DataFrame({"Região": REGIOES,
                        "Passageiros": passageiros,
                        "Necessários": necessarios,
                        "Atuais": x_atual,
                        "Saldo (atuais - necessários)": saldo})
print("Total necessário:", necessarios.sum(), "| Frota disponível:", x_atual.sum())
print("")
tabela2

Total necessário: 72 | Frota disponível: 72



,Região,Passageiros,Necessários,Atuais,Saldo (atuais - necessários)
0,Norte,560,14,12,-2
1,Sul,760,19,20,1
2,Leste,720,18,16,-2
3,Oeste,840,21,24,3


## b) Desequilíbrios entre a distribuição
Identificar possíveis desequilíbrios entre a quantidade de ônibus disponível e a necessidade estimada de cada região:

**Resposta:** O total necessário (72) é **igual** à frota disponível (72). Ou seja, não falta ônibus na cidade, ele está **mal distribuído**:

- **Faltam** ônibus no Norte (2) e no Leste (2);
- **Sobram** ônibus no Sul (1) e no Oeste (3).

Os 4 ônibus que sobram são exatamente os 4 que faltam.

In [58]:
print("Faltam:", {REGIOES[i]: int(-saldo[i]) for i in range(4) if saldo[i] < 0})
print("Sobram:", {REGIOES[i]: int(saldo[i]) for i in range(4) if saldo[i] > 0})

Faltam: {'Norte': 2, 'Leste': 2}
Sobram: {'Sul': 1, 'Oeste': 3}


## c) Política proposta e critérios
**Critérios adotados:**
1. **Preservar a frota total** (72 ônibus), como exige a prefeitura.
2. **Só mover ônibus de quem tem sobra para quem tem falta** (quem não tem sobra não cede veículos).
3. **Mover o mínimo possível** (4 ônibus), para reduzir o custo e a perturbação da operação.
4. Manter as quantidades **inteiras e não negativas**.

**Política:**

| Origem | Destino | Ônibus | Fração da frota da origem |
|---|---|---|---|
| Sul | Leste | 1 | $1/20$ |
| Oeste | Leste | 1 | $1/24$ |
| Oeste | Norte | 2 | $2/24 = 1/12$ |

## d) e e) Transformação linear e matriz associada
Nessa parte, a gente modelou o movimento da frota de ônibus como uma transformação linear $T(\mathbf{x}) = M\mathbf{x}$. , em que

$$M_{ij} = \text{fração da frota da região } j \text{ (origem) que passa a operar na região } i \text{ (destino)}$$

Basicamente, cada coluna da matriz $M$ funciona como um "raio-x" que mostra o destino dos ônibus de uma região de origem (considerando a ordem das colunas: Norte, Sul, Leste e Oeste):   

$$
M = \begin{bmatrix}
1 & 0 & 0 & \tfrac{1}{12}\\[2pt]
0 & \tfrac{19}{20} & 0 & 0\\[2pt]
0 & \tfrac{1}{20} & 1 & \tfrac{1}{24}\\[2pt]
0 & 0 & 0 & \tfrac{7}{8}
\end{bmatrix}
$$

**Significado dos coeficientes:**
* **A Diagonal Principal (Quem fica):** Os elementos da diagonal indicam a fração da frota que continua na mesma região. Por exemplo, no Norte e no Leste $100\%$ dos ônibus ficam, enquanto no Sul ficam $19/20$ e no Oeste $7/8$.
* **Fora da Diagonal (Quem se move):** Os outros valores mostram o fluxo de veículos entre regiões diferentes. Por exemplo, $1/20$ da frota do Sul vai para o Leste, e do Oeste partes vão para o Norte ($1/12$) e para o Leste ($1/24$).
* **Precisão numérica:** Para evitar qualquer tipo de erro de arredondamento com números decimais no computador, utilizamos a classe Fraction, garantindo que todas as contas trabalhem com frações exatas.

In [59]:
M = np.array([[F(1), F(0),     F(0), F(1, 12)],
              [F(0), F(19, 20), F(0), F(0)],
              [F(0), F(1, 20), F(1), F(1, 24)],
              [F(0), F(0),     F(0), F(7, 8)]], dtype=object)

df_matriz = pd.DataFrame(
    [[str(v) for v in linha] for linha in M],
    index=[f"Destino {r}" for r in REGIOES],
    columns=[f"Origem {r}" for r in REGIOES],
)

display(df_matriz)

,Origem Norte,Origem Sul,Origem Leste,Origem Oeste
Destino Norte,1,0,0,1/12
Destino Sul,0,19/20,0,0
Destino Leste,0,1/20,1,1/24
Destino Oeste,0,0,0,7/8


Cada coluna representa a origem (Norte, Sul, Leste, Oeste) e cada linha representa o destino. Ou seja, a matriz inteira é o "mapa" de para onde os ônibus vão.

* Frota do Norte: $\frac{1}{1}$ (ou $100\%$) permanece na região Norte.   
* Frota do Sul: $\frac{19}{20}$ permanece no Sul e $\frac{1}{20}$ vai para o Leste.   
* Frota do Leste: $\frac{1}{1}$ (ou $100\%$) permanece na região Leste.   
* Frota do Oeste: $\frac{7}{8}$ permanece no Oeste, $\frac{1}{12}$ vai para o Norte e $\frac{1}{24}$ vai para o Leste.

## f) Demonstração matemática

Para provar que a política funciona na prática, precisamos verificar duas regras fundamentais: se ela mantém a frota total intacta e se não gera valores negativos.

**1) A transformação preserva o total de ônibus:**
Se a gente somar as frações de cada coluna da matriz $M$, todas dão exatamente 1 (ou seja, todo ônibus que sai de uma região vai para outra ou continua nela, sem sumir nem aparecer do nada):

- Coluna Norte: $1$
- Coluna Sul: $\tfrac{19}{20} + \tfrac{1}{20} = 1$
- Coluna Leste: $1$
- Coluna Oeste: $\tfrac{1}{12} + \tfrac{1}{24} + \tfrac{7}{8} = \tfrac{2}{24} + \tfrac{1}{24} + \tfrac{21}{24} = 1$

Multiplicando isso pela matriz, a soma da frota nova $M\mathbf{x}$ continua exatamente igual à frota antiga $\mathbf{x}$ (que no nosso caso é de 72 ônibus).

**2)** A transformação não produz alocações negativas:
Como todos os números da matriz $M$ e a quantidade de ônibus da frota são positivos ou zero, qualquer conta que a gente faça vai resultar em somas e multiplicações de números maiores ou iguais a zero. Ou seja, é matematicamente impossível aparecer uma quantidade negativa de ônibus.

$$(M\mathbf{x})_i = \sum_{j=1}^{4} m_{ij}\,x_j \;\geq\; 0$$

**3)** Testando com o nosso vetor de frota inicial $\mathbf{x} = [12, 20, 16, 24]^T$, a multiplicação $M\mathbf{x}$ resulta na distribuição exata que as regiões precisam:

$$M\mathbf{x} = \begin{bmatrix} 12 + \tfrac{24}{12}\\[2pt] \tfrac{19}{20}\cdot 20\\[2pt] \tfrac{20}{20} + 16 + \tfrac{24}{24}\\[2pt] \tfrac{7}{8}\cdot 24 \end{bmatrix} = \begin{bmatrix} 14\\ 19\\ 18\\ 21 \end{bmatrix}$$

Assim, provamos que o modelo é totalmente consistente, realista e aplicável ao problema.

In [60]:
# 1) a soma de cada coluna de M é igual a 1 (preservação do total)
print("Verificação da Soma das Colunas:")
for j in range(4):
    soma_coluna = sum(M[i][j] for i in range(4))
    print(f"Coluna {REGIOES[j]}: soma = {soma_coluna}")

# coeficientes são maiores ou iguais a zero
coeficientes_positivos = all(M[i][j] >= 0 for i in range(4) for j in range(4))
print(f"\nTodos os coeficientes são >= 0? {coeficientes_positivos}")

# 3) frota atual (M @ x_atual)
x_novo_frac = [sum(M[i][j] * F(int(x_atual[j])) for j in range(4)) for i in range(4)]
x_novo = np.array([int(v) for v in x_novo_frac])

print(f"\nTotal de ônibus antes: {x_atual.sum()}")
print(f"Total de ônibus depois: {x_novo.sum()}")

df_resultado = pd.DataFrame({
    "Região": REGIOES,
    "Antes (Frota Atual)": x_atual,
    "Depois (Calculado)": x_novo,
    "Necessários": necessarios
})

display(df_resultado)

Verificação da Soma das Colunas:
Coluna Norte: soma = 1
Coluna Sul: soma = 1
Coluna Leste: soma = 1
Coluna Oeste: soma = 1

Todos os coeficientes são >= 0? True

Total de ônibus antes: 72
Total de ônibus depois: 72


,Região,Antes (Frota Atual),Depois (Calculado),Necessários
0,Norte,12,14,14
1,Sul,20,19,19
2,Leste,16,18,18
3,Oeste,24,21,21


## Avaliação do resultado
Comparamos o atendimento à demanda no pico antes e depois da política: a % da demanda atendida é $\min\!\left(1, \dfrac{\text{ônibus}\times 40}{\text{passageiros}}\right)$.

In [61]:
atend_antes = np.minimum(1, x_atual * capacidade / passageiros)
atend_depois = np.minimum(1, x_novo * capacidade / passageiros)

avaliacao = pd.DataFrame({"Região": REGIOES,
                          "Atendimento antes (%)": (100 * atend_antes).round(1),
                          "Atendimento depois (%)": (100 * atend_depois).round(1)})
sem_vaga_antes = int(np.maximum(0, passageiros - x_atual * capacidade).sum())
sem_vaga_depois = int(np.maximum(0, passageiros - x_novo * capacidade).sum())

print(f"Passageiros sem vaga no pico: Antes: {sem_vaga_antes} -> Depois: {sem_vaga_depois}")
print("")
avaliacao

Passageiros sem vaga no pico: Antes: 160 -> Depois: 0



,Região,Atendimento antes (%),Atendimento depois (%)
0,Norte,85.7,100.0
1,Sul,100.0,100.0
2,Leste,88.9,100.0
3,Oeste,100.0,100.0


## Conclusão

* As informações do Relatório I determinam uma única distribuição atual (Norte: 12, Sul: 20, Leste: 16 e Oeste: 24), garantida pelo fato de o determinante da matriz ser diferente de zero ($\det(A) = 3 \neq 0$).
* Diagnóstico da frota: O total de 72 ônibus é suficiente para atender à demanda do horário de pico, mas a frota estava mal distribuída entre as regiões (faltavam veículos no Norte e no Leste, enquanto havia excesso no Sul e no Oeste).
* A transformação linear $T(\mathbf{x}) = M\mathbf{x}$ resolveu o problema movimentando apenas 4 ônibus, preservando o total da frota (soma das colunas igual a 1), sem gerar alocações negativas e zerando o déficit de passageiros.
* Limitações do estudo que indentificamos: O modelo baseia-se em dados hipotéticos de um único horário de pico deum dia e na matriz de transição atual, desconsiderando variáveis logísticas complexas, como distâncias reais entre as regiões e custos operacionais de deslocamento para as mudanças dos ônibus.